In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install',
                       'matplotlib', 'networkx', 'numpy', 'ipywidgets'],
                      stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

import networkx as nx
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import numpy as np
import heapq, math, warnings
from collections import deque
warnings.filterwarnings('ignore')
plt.rcParams['figure.facecolor'] = '#f5f5f5'
plt.rcParams['axes.facecolor']   = '#f5f5f5'

# ══════════════════════════════════════════════════════════════
#  GRAPH DATA
# ══════════════════════════════════════════════════════════════

EDGES = [
    ('A','B',4),('A','C',3),('B','D',2),('B','E',5),
    ('C','E',1),('C','F',6),('D','G',3),('E','G',4),
    ('E','H',2),('F','H',3),('G','I',2),('G','J',5),
    ('H','I',1),('I','J',3)
]
POS = {
    'A':(0,2),'B':(1,3),'C':(1,1),'D':(2,4),
    'E':(2,2),'F':(2,0),'G':(3,3.5),'H':(3,1),
    'I':(4,2.5),'J':(5,3)
}

def heuristic(node, goal):
    x1,y1=POS[node]; x2,y2=POS[goal]
    return math.sqrt((x2-x1)**2+(y2-y1)**2)

def build_graph(edges):
    g={}
    for u,v,w in edges:
        g.setdefault(u,[]).append((v,w))
        g.setdefault(v,[]).append((u,w))
    return g

GRAPH = build_graph(EDGES)

# ══════════════════════════════════════════════════════════════
#  DRAW HELPER
# ══════════════════════════════════════════════════════════════

COLORS = {
    'start':'#D85A30','goal':'#7F77DD','path':'#E24B4A',
    'visited':'#1D9E75','frontier':'#EF9F27','default':'#378ADD'
}

def draw_graph(ax, visited=(), frontier=(), path=(), start=None, goal=None, title=''):
    G=nx.Graph()
    for u,v,w in EDGES: G.add_edge(u,v,weight=w)
    visited=set(visited); frontier=set(frontier); path=list(path)
    path_edges={(path[i],path[i+1]) for i in range(len(path)-1)}
    path_edges|={(b,a) for a,b in path_edges}
    node_colors=[
        COLORS['start']    if n==start    else
        COLORS['goal']     if n==goal     else
        COLORS['path']     if n in path   else
        COLORS['visited']  if n in visited  else
        COLORS['frontier'] if n in frontier else
        COLORS['default']
        for n in G.nodes()
    ]
    edge_colors=['#E24B4A' if (u,v) in path_edges else '#cccccc' for u,v in G.edges()]
    edge_widths=[3.0       if (u,v) in path_edges else 1.2       for u,v in G.edges()]
    nx.draw_networkx(G,pos=POS,ax=ax,node_color=node_colors,node_size=700,
                     edge_color=edge_colors,width=edge_widths,
                     font_color='white',font_size=11,font_weight='bold',with_labels=True)
    nx.draw_networkx_edge_labels(G,pos=POS,edge_labels=nx.get_edge_attributes(G,'weight'),
                                  ax=ax,font_size=8,font_color='#555')
    ax.set_title(title,fontsize=10,fontweight='bold')
    ax.axis('off')

# ══════════════════════════════════════════════════════════════
#  SEARCH ALGORITHMS
# ══════════════════════════════════════════════════════════════

def bfs(graph, start, goal):
    queue=deque([(start,[start])]); visited=set([start]); steps=[]
    while queue:
        node,path=queue.popleft()
        steps.append(dict(cur=node,visited=set(visited),
                          frontier={n for n,_ in queue},path=[]))
        if node==goal: steps[-1]['path']=path; return path,steps
        for nb,_ in sorted(graph.get(node,[])):
            if nb not in visited:
                visited.add(nb); queue.append((nb,path+[nb]))
    return None,steps

def dfs(graph, start, goal):
    stack=[(start,[start])]; visited=set(); steps=[]
    while stack:
        node,path=stack.pop()
        if node in visited: continue
        visited.add(node)
        steps.append(dict(cur=node,visited=set(visited),
                          frontier={n for n,_ in stack},path=[]))
        if node==goal: steps[-1]['path']=path; return path,steps
        for nb,_ in sorted(graph.get(node,[]),reverse=True):
            if nb not in visited: stack.append((nb,path+[nb]))
    return None,steps

def ucs(graph, start, goal):
    heap=[(0,start,[start])]; visited={}; steps=[]
    while heap:
        cost,node,path=heapq.heappop(heap)
        if node in visited: continue
        visited[node]=cost
        steps.append(dict(cur=node,cost=cost,visited=dict(visited),
                          frontier=[(c,n) for c,n,_ in heap],path=[]))
        if node==goal: steps[-1]['path']=path; return path,cost,steps
        for nb,w in sorted(graph.get(node,[])):
            if nb not in visited: heapq.heappush(heap,(cost+w,nb,path+[nb]))
    return None,float('inf'),steps

def astar(graph, start, goal):
    h0=heuristic(start,goal)
    heap=[(h0,0,start,[start])]; visited={}; steps=[]
    while heap:
        f,g,node,path=heapq.heappop(heap)
        if node in visited: continue
        visited[node]=g
        steps.append(dict(cur=node,g=g,h=round(heuristic(node,goal),2),
                          f=round(f,2),visited=dict(visited),
                          frontier=[(fc,n) for fc,_,n,_ in heap],path=[]))
        if node==goal: steps[-1]['path']=path; return path,g,steps
        for nb,w in sorted(graph.get(node,[])):
            if nb not in visited:
                g2=g+w; f2=g2+heuristic(nb,goal)
                heapq.heappush(heap,(f2,g2,nb,path+[nb]))
    return None,float('inf'),steps

# ══════════════════════════════════════════════════════════════
#  UI
# ══════════════════════════════════════════════════════════════

display(HTML("""
<div style='background:linear-gradient(135deg,#1a237e,#283593);
             color:white;padding:18px 26px;border-radius:12px;margin-bottom:14px'>
  <h2 style='margin:0;font-size:20px'>Graph Search Algorithm Visualizer</h2>
  <p style='margin:5px 0 0;opacity:.85;font-size:13px'>
    BFS &nbsp;|&nbsp; DFS &nbsp;|&nbsp; UCS &nbsp;|&nbsp; A*
  </p>
</div>
"""))

algo_toggle = widgets.ToggleButtons(
    options=['BFS','DFS','UCS','A*'],
    button_style='info',
    style={'button_width':'110px'}
)
node_labels = sorted(GRAPH.keys())
start_dd = widgets.Dropdown(options=node_labels, value='A', description='Start:',
                            style={'description_width':'45px'},
                            layout=widgets.Layout(width='140px'))
goal_dd  = widgets.Dropdown(options=node_labels, value='J', description='Goal:',
                            style={'description_width':'45px'},
                            layout=widgets.Layout(width='140px'))
run_btn  = widgets.Button(description='Run', button_style='success',
                          layout=widgets.Layout(width='90px'))

graph_out = widgets.Output()
log_out   = widgets.Output()

with graph_out:
    fig,ax=plt.subplots(figsize=(10,5))
    draw_graph(ax, start='A', goal='J', title='Pick start & goal nodes, choose an algorithm, then click Run')
    plt.tight_layout(); plt.show()

def run_search(b):
    s,g  = start_dd.value, goal_dd.value
    algo = algo_toggle.value
    if s==g:
        with log_out: clear_output(); print('Start and goal must be different nodes.')
        return

    cost_str=''; is_cost=False
    if   algo=='BFS': path,steps=bfs(GRAPH,s,g)
    elif algo=='DFS': path,steps=dfs(GRAPH,s,g)
    elif algo=='UCS': path,cost,steps=ucs(GRAPH,s,g);   cost_str=f'  |  Cost: {cost}';         is_cost=True
    else:             path,cost,steps=astar(GRAPH,s,g); cost_str=f'  |  Cost: {round(cost,2)}'; is_cost=True

    with graph_out:
        clear_output(wait=True)
        n=len(steps)
        idxs=list(dict.fromkeys([int(i*(n-1)/7) for i in range(7)]+[n-1])) if n>=8 else list(range(n))
        idxs=idxs[:8]
        cols=4; rows=max(1,math.ceil(len(idxs)/cols))
        fig,axes=plt.subplots(rows,cols,figsize=(16,rows*4.2))
        axes=np.array(axes).flatten()
        for ax in axes: ax.axis('off')
        for i,idx in enumerate(idxs):
            st=steps[idx]
            vis=set(st['visited'].keys()) if isinstance(st['visited'],dict) else st['visited']
            fr =[nn for _,nn in st['frontier']] if is_cost else st['frontier']
            extra=''
            if 'f' in st:      extra=f'  f={st["f"]}  g={st["g"]}  h={st["h"]}'
            elif 'cost' in st: extra=f'  cost={st["cost"]}'
            draw_graph(axes[i],visited=vis,frontier=fr,
                       path=st.get('path',[]),start=s,goal=g,
                       title=f'Step {idx+1} - {st["cur"]}'+extra)
        path_str=' -> '.join(path) if path else 'None'
        plt.suptitle(f'{algo}:  {s} -> {g}  |  Path: {path_str}{cost_str}  |  Nodes expanded: {len(steps)}',
                     fontsize=12,fontweight='bold',y=1.01)
        plt.tight_layout(); plt.show()

    with log_out:
        clear_output(wait=True)
        sep='─'*66
        print(sep)
        print(f'{algo} Traversal Log  |  {s} -> {g}')
        print(sep)
        for i,st in enumerate(steps):
            tag=' <- GOAL' if st.get('path') else ''
            if 'f' in st:
                vis_str=', '.join(f'{k}(g={v})' for k,v in sorted(st['visited'].items()))
                print(f'Step {i+1:2d}: {st["cur"]}  f={st["f"]:5.2f}  g={st["g"]}  h={st["h"]:4.2f}  |  {vis_str}{tag}')
            elif 'cost' in st:
                vis_str=', '.join(f'{k}({v})' for k,v in sorted(st['visited'].items()))
                print(f'Step {i+1:2d}: {st["cur"]}  cost={st["cost"]:2d}  |  {vis_str}{tag}')
            else:
                print(f'Step {i+1:2d}: Visit {st["cur"]}  |  Visited: {{", ".join(sorted(st["visited"]))}}  {tag}')
        print(sep)
        print(f'Path: {" -> ".join(path) if path else "No path found"}{cost_str}')

run_btn.on_click(run_search)

display(HTML('<b>Algorithm:</b>'))
display(algo_toggle)
display(widgets.HBox([start_dd, goal_dd, run_btn],
                     layout=widgets.Layout(gap='10px', margin='8px 0')))
display(HTML("""
<div style='display:flex;gap:10px;flex-wrap:wrap;margin:6px 0 12px;font-size:13px'>
  <b style='background:#D85A30;padding:2px 10px;border-radius:4px;color:white'>Start</b>
  <b style='background:#7F77DD;padding:2px 10px;border-radius:4px;color:white'>Goal</b>
  <b style='background:#EF9F27;padding:2px 10px;border-radius:4px;color:white'>Frontier</b>
  <b style='background:#1D9E75;padding:2px 10px;border-radius:4px;color:white'>Visited</b>
  <b style='background:#E24B4A;padding:2px 10px;border-radius:4px;color:white'>Final Path</b>
  <b style='background:#378ADD;padding:2px 10px;border-radius:4px;color:white'>Unvisited</b>
</div>
"""))
display(graph_out)
display(HTML('<b style="font-size:13px">Traversal Log:</b>'))
display(log_out)
display(HTML("""
<div style='background:#1e3a5f;border-left:4px solid #5c9bd6;
            padding:10px 14px;border-radius:6px;margin-top:12px;font-size:13px;color:#e8f0fe'>
  <b style='color:#ffffff'>BFS</b> <span style='color:#e8f0fe'>— Queue, explores level by level, fewest hops guaranteed</span> &nbsp;|&nbsp;
  <b style='color:#ffffff'>DFS</b> <span style='color:#e8f0fe'>— Stack, dives deep first, memory efficient, not always shortest</span> &nbsp;|&nbsp;
  <b style='color:#ffffff'>UCS</b> <span style='color:#e8f0fe'>— Min-heap on g(n), expands lowest cumulative cost, weighted optimal</span> &nbsp;|&nbsp;
  <b style='color:#ffffff'>A*</b>  <span style='color:#e8f0fe'>— Min-heap on f(n)=g(n)+h(n), heuristic-guided, faster than UCS</span>
</div>
"""))